# Module 7 • Hugging Face and Pretrained Transformer Workflows

# Lesson 36 • Fine-Tuning Pretrained Transformers for Text Classification

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 180–220 minutes  
**Execution target:** CPU by default

---

## Scope

This lesson develops a complete supervised fine-tuning workflow for text
classification with pretrained Transformer models. It covers dataset design,
label mapping, tokenizer integration, frozen baselines, full fine-tuning,
checkpoint selection, learning-rate control, class metrics, confidence analysis,
error analysis, statistical comparison, reproducibility, and deployment
considerations.

The notebook includes:

1. a complete offline CPU experiment that executes without internet access;
2. optional Hugging Face fine-tuning cells that run only when the required
   packages and a cached or downloadable checkpoint are available.

## Learning Objectives

After completing this lesson, the learner should be able to:

- formulate a supervised text-classification task;
- build deterministic train, validation, and test splits;
- define label-to-ID and ID-to-label mappings;
- tokenize text for a pretrained encoder;
- distinguish frozen-feature extraction from full fine-tuning;
- create dynamic padded batches;
- train a classifier with validation-based checkpoint selection;
- calculate accuracy, macro F1, per-class metrics, and confusion matrices;
- inspect confidence and error patterns;
- compare two models with bootstrap confidence intervals;
- structure a Hugging Face AutoModel fine-tuning workflow;
- explain learning-rate, batch-size, truncation, and memory trade-offs;
- preserve reproducibility metadata;
- assess Arabic and multilingual fine-tuning requirements.

## Table of Contents

1. Fine-Tuning as Transfer Learning
2. Task Formulation
3. Dataset and Labels
4. Train, Validation, and Test Splits
5. Label Mapping
6. Tokenization Strategy
7. Padding and Truncation
8. Frozen Baseline
9. Full Fine-Tuning
10. Offline Dataset
11. Offline Tokenizer
12. Dataset and Collation
13. Transformer Encoder
14. Frozen Classifier
15. Fine-Tuned Classifier
16. Training Utilities
17. Frozen Training
18. Fine-Tuned Training
19. Learning Curves
20. Validation-Based Selection
21. Test Evaluation
22. Confusion Matrix
23. Per-Class Metrics
24. Confidence Analysis
25. Error Analysis
26. Bootstrap Confidence Intervals
27. Paired Model Comparison
28. Calibration Discussion
29. Optional Hugging Face Setup
30. Optional Tokenization
31. Optional AutoModel Fine-Tuning
32. Optional Trainer Workflow
33. Checkpoint Management
34. Learning-Rate and Batch-Size Decisions
35. Class Imbalance
36. Domain Shift
37. Arabic and Multilingual Considerations
38. Reproducibility and Reporting
39. Knowledge Check
40. Exercises
41. Summary and Next Lesson

# 1. Fine-Tuning as Transfer Learning

Fine-tuning adapts a pretrained model to a labeled downstream task.

A sequence-classification model contains:

- a pretrained tokenizer;
- a pretrained Transformer encoder;
- a task-specific classification head.

In [ ]:
import copy
import importlib.util
import math
import platform
import random
import re
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

transfer_learning_components = pd.DataFrame(
    [
        ("Tokenizer", "converts text to model inputs"),
        ("Encoder", "provides pretrained contextual representations"),
        ("Classification head", "maps representations to labels"),
    ],
    columns=["Component", "Role"],
)

transfer_learning_components

# 2. Task Formulation

This lesson uses single-label, multi-class text classification.

Every example has exactly one label.

In [ ]:
task_definition = pd.Series(
    {
        "input": "one text sequence",
        "output": "one class label",
        "classes": 4,
        "loss": "cross-entropy",
        "primary metric": "macro F1",
    }
)

task_definition

Macro F1 gives equal weight to every class and is useful when class performance
should be treated equally.

# 3. Dataset and Labels

In [ ]:
records = [
    ("doctor treats patient in hospital", "health"),
    ("nurse provides medicine to patient", "health"),
    ("patient visits clinic for diagnosis", "health"),
    ("hospital schedules medical treatment", "health"),
    ("exercise supports long term health", "health"),
    ("nutrition improves patient recovery", "health"),
    ("doctor reviews the medical report", "health"),
    ("clinic provides emergency service", "health"),
    ("nurse helps the patient today", "health"),
    ("medicine reduces the health problem", "health"),
    ("hospital needs experienced doctors", "health"),
    ("patient requests treatment information", "health"),
    ("medical team monitors patient recovery", "health"),
    ("doctor confirms the diagnosis later", "health"),
    ("clinic updates the treatment plan", "health"),
    ("patient receives medicine after examination", "health"),

    ("bank approves customer loan", "finance"),
    ("invoice contains payment charge", "finance"),
    ("customer requests card refund", "finance"),
    ("billing account has a problem", "finance"),
    ("loan interest increased today", "finance"),
    ("bank transfers money safely", "finance"),
    ("payment failed on the card", "finance"),
    ("refund request remains pending", "finance"),
    ("invoice price is incorrect", "finance"),
    ("customer updates bank account", "finance"),
    ("billing service changed the charge", "finance"),
    ("loan payment needs approval", "finance"),
    ("bank reviews the financial request", "finance"),
    ("customer receives refund after review", "finance"),
    ("payment system confirms the transaction", "finance"),
    ("card account shows an extra charge", "finance"),

    ("software update caused an error", "technology"),
    ("application cannot reach the server", "technology"),
    ("network upload failed today", "technology"),
    ("computer needs a system update", "technology"),
    ("device cannot install the software", "technology"),
    ("server lost important data", "technology"),
    ("application displays a network error", "technology"),
    ("computer connects to the server", "technology"),
    ("upload request failed again", "technology"),
    ("system update needs technical help", "technology"),
    ("device reports a software problem", "technology"),
    ("network service is unavailable", "technology"),
    ("server restarts after the update", "technology"),
    ("application recovers after installation", "technology"),
    ("computer stores data on server", "technology"),
    ("network error interrupts the upload", "technology"),

    ("flight arrives at airport", "travel"),
    ("tourist books hotel reservation", "travel"),
    ("airport lost passenger luggage", "travel"),
    ("travel ticket changed today", "travel"),
    ("flight delay affects the journey", "travel"),
    ("hotel reservation needs an update", "travel"),
    ("tourist visits the city museum", "travel"),
    ("beach trip starts tomorrow", "travel"),
    ("airport changes the flight gate", "travel"),
    ("passenger requests travel information", "travel"),
    ("journey includes a hotel stay", "travel"),
    ("ticket service reports a delay", "travel"),
    ("tourist reaches airport before departure", "travel"),
    ("passenger collects luggage after arrival", "travel"),
    ("hotel confirms the reservation", "travel"),
    ("flight continues after a short delay", "travel"),
]

dataset = pd.DataFrame(
    records,
    columns=["text", "label"],
)

dataset["label"].value_counts()

# 4. Train, Validation, and Test Splits

- training data updates parameters;
- validation data selects checkpoints and hyperparameters;
- test data is used once for final evaluation.

In [ ]:
(
    X_train_full,
    X_test,
    y_train_full,
    y_test,
) = train_test_split(
    dataset["text"],
    dataset["label"],
    test_size=0.25,
    random_state=42,
    stratify=dataset["label"],
)

(
    X_train,
    X_validation,
    y_train,
    y_validation,
) = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

split_summary = pd.Series(
    {
        "training": len(X_train),
        "validation": len(X_validation),
        "test": len(X_test),
    }
)

split_summary

# 5. Label Mapping

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(y_train)

label2id = {
    label: int(index)
    for index, label
    in enumerate(label_encoder.classes_)
}

id2label = {
    index: label
    for label, index
    in label2id.items()
}

pd.DataFrame(
    {
        "label": list(label2id.keys()),
        "id": list(label2id.values()),
    }
)

The same label mapping must be reused for training, evaluation, saving, and
deployment.

# 6. Tokenization Strategy

Pretrained models require the tokenizer associated with their checkpoint.

Tokenization decisions include:

- maximum length;
- truncation;
- padding;
- special tokens;
- return format;
- language-specific normalization.

# 7. Padding and Truncation

Dynamic padding usually wastes less computation than padding every example to a
global maximum.

Truncation should be justified by sequence-length analysis.

In [ ]:
text_lengths = dataset["text"].map(
    lambda text: len(text.split())
)

text_lengths.describe()

# 8. Frozen Baseline

A frozen baseline keeps encoder parameters fixed and trains only the classification
head.

Benefits:

- fast;
- low memory;
- useful as a transfer-learning baseline.

Limitation:

- reduced task adaptation.

# 9. Full Fine-Tuning

Full fine-tuning updates encoder and classifier parameters.

It usually requires:

- a smaller learning rate;
- careful validation;
- regularization;
- checkpoint selection.

In [ ]:
strategy_comparison = pd.DataFrame(
    [
        ("Frozen", "head only", "higher", "lower"),
        ("Fine-tuned", "encoder + head", "lower", "higher"),
    ],
    columns=[
        "Strategy",
        "Updated parameters",
        "Typical learning rate",
        "Memory use",
    ],
)

strategy_comparison

# 10. Offline Dataset

The remaining executable core uses a local Transformer encoder. This mirrors the
fine-tuning workflow while avoiding downloads.

# 11. Offline Tokenizer

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


counts = Counter(
    token
    for text in X_train
    for token in tokenize(text)
)

vocabulary = [
    "<PAD>",
    "<UNK>",
    "<CLS>",
] + sorted(counts)

token_to_index = {
    token: index
    for index, token in enumerate(vocabulary)
}

PAD_ID = token_to_index["<PAD>"]
UNK_ID = token_to_index["<UNK>"]
CLS_ID = token_to_index["<CLS>"]

print("Vocabulary size:", len(vocabulary))

# 12. Dataset and Collation

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(
        self,
        texts,
        labels,
    ):
        self.texts = list(texts)
        self.labels = label_encoder.transform(
            list(labels)
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        token_ids = [
            CLS_ID
        ] + [
            token_to_index.get(
                token,
                UNK_ID,
            )
            for token in tokenize(
                self.texts[index]
            )
        ]

        return {
            "input_ids": torch.tensor(
                token_ids,
                dtype=torch.long,
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long,
            ),
            "text": self.texts[index],
        }


def collate_batch(batch):
    maximum_length = max(
        len(item["input_ids"])
        for item in batch
    )

    input_ids = torch.full(
        (
            len(batch),
            maximum_length,
        ),
        PAD_ID,
        dtype=torch.long,
    )

    labels = []

    for row, item in enumerate(batch):
        sequence = item["input_ids"]

        input_ids[
            row,
            :len(sequence),
        ] = sequence

        labels.append(item["label"])

    return {
        "input_ids": input_ids,
        "attention_mask": (
            input_ids != PAD_ID
        ).long(),
        "padding_mask": (
            input_ids == PAD_ID
        ),
        "labels": torch.stack(labels),
        "texts": [
            item["text"]
            for item in batch
        ],
    }

In [ ]:
train_dataset = ClassificationDataset(
    X_train,
    y_train,
)
validation_dataset = ClassificationDataset(
    X_validation,
    y_validation,
)
test_dataset = ClassificationDataset(
    X_test,
    y_test,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(42),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_batch,
)

sample_batch = next(iter(train_loader))

print(
    sample_batch["input_ids"].shape,
    sample_batch["attention_mask"].shape,
)

# 13. Transformer Encoder

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(
        self,
        model_dimension: int,
        maximum_length: int = 128,
    ):
        super().__init__()

        encoding = torch.zeros(
            maximum_length,
            model_dimension,
        )

        positions = torch.arange(
            maximum_length,
            dtype=torch.float32,
        ).unsqueeze(1)

        rates = torch.exp(
            torch.arange(
                0,
                model_dimension,
                2,
                dtype=torch.float32,
            )
            * (
                -math.log(10000.0)
                / model_dimension
            )
        )

        encoding[:, 0::2] = torch.sin(
            positions * rates
        )
        encoding[:, 1::2] = torch.cos(
            positions * rates
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
        )

    def forward(
        self,
        embeddings: torch.Tensor,
    ) -> torch.Tensor:
        return (
            embeddings
            + self.encoding[
                :,
                :embeddings.size(1),
                :,
            ]
        )


class TinyEncoder(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        model_dimension: int = 32,
        head_count: int = 4,
        layer_count: int = 2,
        feed_forward_dimension: int = 64,
        dropout: float = 0.15,
    ):
        super().__init__()

        self.model_dimension = model_dimension

        self.embedding = nn.Embedding(
            vocabulary_size,
            model_dimension,
            padding_idx=PAD_ID,
        )

        self.position = PositionalEncoding(
            model_dimension
        )

        layer = nn.TransformerEncoderLayer(
            d_model=model_dimension,
            nhead=head_count,
            dim_feedforward=(
                feed_forward_dimension
            ),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=layer_count,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        embeddings = (
            self.embedding(input_ids)
            * math.sqrt(
                self.model_dimension
            )
        )

        return self.encoder(
            self.position(embeddings),
            src_key_padding_mask=(
                padding_mask
            ),
        )

# 14. Frozen Classifier

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(
        self,
        encoder: TinyEncoder,
        class_count: int,
        freeze_encoder: bool,
    ):
        super().__init__()

        self.encoder = encoder

        if freeze_encoder:
            for parameter in (
                self.encoder.parameters()
            ):
                parameter.requires_grad = False

        self.dropout = nn.Dropout(0.15)

        self.classifier = nn.Linear(
            encoder.model_dimension,
            class_count,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ):
        hidden_states = self.encoder(
            input_ids,
            padding_mask,
        )

        representation = hidden_states[
            :,
            0,
            :,
        ]

        logits = self.classifier(
            self.dropout(
                representation
            )
        )

        return {
            "logits": logits,
            "representation": representation,
        }


DEVICE = torch.device("cpu")

torch.manual_seed(42)

base_encoder = TinyEncoder(
    vocabulary_size=len(vocabulary)
).to(DEVICE)

initial_encoder_state = copy.deepcopy(
    base_encoder.state_dict()
)

# 15. Fine-Tuned Classifier

In [ ]:
def create_classifier(
    freeze_encoder: bool,
) -> SequenceClassifier:
    encoder = TinyEncoder(
        vocabulary_size=len(vocabulary)
    )

    encoder.load_state_dict(
        initial_encoder_state
    )

    return SequenceClassifier(
        encoder=encoder,
        class_count=len(
            label_encoder.classes_
        ),
        freeze_encoder=freeze_encoder,
    ).to(DEVICE)


frozen_model = create_classifier(
    freeze_encoder=True
)

fine_tuned_model = create_classifier(
    freeze_encoder=False
)

print(
    "Frozen trainable parameters:",
    sum(
        parameter.numel()
        for parameter in frozen_model.parameters()
        if parameter.requires_grad
    ),
)

print(
    "Fine-tuned trainable parameters:",
    sum(
        parameter.numel()
        for parameter in fine_tuned_model.parameters()
        if parameter.requires_grad
    ),
)

# 16. Training Utilities

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


loss_function = nn.CrossEntropyLoss()


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
):
    model.eval()

    losses = []
    labels_all = []
    predictions_all = []
    probabilities_all = []
    texts_all = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch[
                "input_ids"
            ].to(DEVICE)

            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)

            labels = batch[
                "labels"
            ].to(DEVICE)

            output = model(
                input_ids,
                padding_mask,
            )

            loss = loss_function(
                output["logits"],
                labels,
            )

            probabilities = torch.softmax(
                output["logits"],
                dim=1,
            )

            predictions = probabilities.argmax(
                dim=1
            )

            losses.append(
                float(loss.item())
            )
            labels_all.extend(
                labels.cpu().tolist()
            )
            predictions_all.extend(
                predictions.cpu().tolist()
            )
            probabilities_all.extend(
                probabilities.cpu().tolist()
            )
            texts_all.extend(
                batch["texts"]
            )

    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(
            labels_all,
            predictions_all,
        ),
        "macro_f1": f1_score(
            labels_all,
            predictions_all,
            average="macro",
        ),
        "labels": np.asarray(labels_all),
        "predictions": np.asarray(
            predictions_all
        ),
        "probabilities": np.asarray(
            probabilities_all
        ),
        "texts": texts_all,
    }


def train_model(
    model: nn.Module,
    epochs: int,
    learning_rate: float,
    patience: int = 8,
):
    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.Adam(
        trainable_parameters,
        lr=learning_rate,
        weight_decay=1e-4,
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )
    best_validation_f1 = -1.0
    without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()

        training_losses = []
        gradient_norms = []

        for batch in train_loader:
            input_ids = batch[
                "input_ids"
            ].to(DEVICE)

            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)

            labels = batch[
                "labels"
            ].to(DEVICE)

            optimizer.zero_grad()

            output = model(
                input_ids,
                padding_mask,
            )

            loss = loss_function(
                output["logits"],
                labels,
            )

            loss.backward()

            gradient_norm = clip_grad_norm_(
                trainable_parameters,
                max_norm=5.0,
            )

            optimizer.step()

            training_losses.append(
                float(loss.item())
            )
            gradient_norms.append(
                float(gradient_norm)
            )

        validation_metrics = evaluate_model(
            model,
            validation_loader,
        )

        history.append(
            {
                "epoch": epoch,
                "training_loss": float(
                    np.mean(training_losses)
                ),
                "validation_loss": (
                    validation_metrics["loss"]
                ),
                "validation_accuracy": (
                    validation_metrics[
                        "accuracy"
                    ]
                ),
                "validation_macro_f1": (
                    validation_metrics[
                        "macro_f1"
                    ]
                ),
                "gradient_norm": float(
                    np.mean(gradient_norms)
                ),
            }
        )

        if (
            validation_metrics["macro_f1"]
            > best_validation_f1
            + 1e-6
        ):
            best_validation_f1 = (
                validation_metrics[
                    "macro_f1"
                ]
            )
            best_state = copy.deepcopy(
                model.state_dict()
            )
            without_improvement = 0
        else:
            without_improvement += 1

        if without_improvement >= patience:
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

# 17. Frozen Training

In [ ]:
set_seed(42)

(
    trained_frozen_model,
    frozen_history,
) = train_model(
    frozen_model,
    epochs=35,
    learning_rate=0.01,
)

print(
    "Frozen best validation F1:",
    round(
        frozen_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

# 18. Fine-Tuned Training

In [ ]:
set_seed(42)

(
    trained_fine_tuned_model,
    fine_tuned_history,
) = train_model(
    fine_tuned_model,
    epochs=35,
    learning_rate=0.002,
)

print(
    "Fine-tuned best validation F1:",
    round(
        fine_tuned_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

# 19. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    frozen_history["epoch"],
    frozen_history[
        "validation_loss"
    ],
    label="Frozen",
)
plt.plot(
    fine_tuned_history["epoch"],
    fine_tuned_history[
        "validation_loss"
    ],
    label="Fine-tuned",
)
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Frozen Versus Fine-Tuned Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    frozen_history["epoch"],
    frozen_history[
        "validation_macro_f1"
    ],
    label="Frozen",
)
plt.plot(
    fine_tuned_history["epoch"],
    fine_tuned_history[
        "validation_macro_f1"
    ],
    label="Fine-tuned",
)
plt.xlabel("Epoch")
plt.ylabel("Validation macro F1")
plt.title("Frozen Versus Fine-Tuned Macro F1")
plt.legend()
plt.tight_layout()
plt.show()

# 20. Validation-Based Selection

In [ ]:
frozen_validation = evaluate_model(
    trained_frozen_model,
    validation_loader,
)

fine_tuned_validation = evaluate_model(
    trained_fine_tuned_model,
    validation_loader,
)

validation_comparison = pd.DataFrame(
    [
        (
            "Frozen",
            frozen_validation["accuracy"],
            frozen_validation["macro_f1"],
        ),
        (
            "Fine-tuned",
            fine_tuned_validation["accuracy"],
            fine_tuned_validation["macro_f1"],
        ),
    ],
    columns=[
        "Model",
        "Validation accuracy",
        "Validation macro F1",
    ],
)

validation_comparison

In [ ]:
if (
    fine_tuned_validation["macro_f1"]
    >= frozen_validation["macro_f1"]
):
    selected_model_name = "Fine-tuned"
    selected_model = trained_fine_tuned_model
else:
    selected_model_name = "Frozen"
    selected_model = trained_frozen_model

print("Selected model:", selected_model_name)

# 21. Test Evaluation

In [ ]:
test_metrics = evaluate_model(
    selected_model,
    test_loader,
)

actual_labels = label_encoder.inverse_transform(
    test_metrics["labels"]
)

predicted_labels = label_encoder.inverse_transform(
    test_metrics["predictions"]
)

print(
    "Test accuracy:",
    round(test_metrics["accuracy"], 3),
)
print(
    "Test macro F1:",
    round(test_metrics["macro_f1"], 3),
)
print()
print(
    classification_report(
        actual_labels,
        predicted_labels,
        zero_division=0,
    )
)

# 22. Confusion Matrix

In [ ]:
class_names = list(
    label_encoder.classes_
)

matrix = confusion_matrix(
    actual_labels,
    predicted_labels,
    labels=class_names,
)

pd.DataFrame(
    matrix,
    index=[
        f"actual_{label}"
        for label in class_names
    ],
    columns=[
        f"predicted_{label}"
        for label in class_names
    ],
)

# 23. Per-Class Metrics

In [ ]:
(
    precision,
    recall,
    f1_values,
    support,
) = precision_recall_fscore_support(
    actual_labels,
    predicted_labels,
    labels=class_names,
    zero_division=0,
)

per_class_metrics = pd.DataFrame(
    {
        "class": class_names,
        "precision": precision,
        "recall": recall,
        "f1": f1_values,
        "support": support,
    }
)

per_class_metrics

# 24. Confidence Analysis

In [ ]:
confidence = test_metrics[
    "probabilities"
].max(axis=1)

confidence_frame = pd.DataFrame(
    {
        "text": test_metrics["texts"],
        "actual": actual_labels,
        "predicted": predicted_labels,
        "confidence": confidence,
    }
)

confidence_frame["correct"] = (
    confidence_frame["actual"]
    == confidence_frame["predicted"]
)

confidence_frame.groupby(
    "correct"
)["confidence"].describe()

High confidence does not guarantee correctness.

# 25. Error Analysis

In [ ]:
error_frame = confidence_frame[
    ~confidence_frame["correct"]
].sort_values(
    "confidence",
    ascending=False,
)

error_frame

Error categories to inspect:

- lexical ambiguity;
- domain overlap;
- unseen vocabulary;
- short inputs;
- class-specific underperformance;
- overconfident mistakes.

# 26. Bootstrap Confidence Intervals

Bootstrap resampling estimates uncertainty in test metrics.

In [ ]:
def bootstrap_metric_interval(
    labels: np.ndarray,
    predictions: np.ndarray,
    metric_function,
    repetitions: int = 1000,
    confidence_level: float = 0.95,
    seed: int = 42,
) -> tuple[float, float, float]:
    generator = np.random.default_rng(
        seed
    )

    values = []
    sample_size = len(labels)

    for _ in range(repetitions):
        indices = generator.integers(
            0,
            sample_size,
            size=sample_size,
        )

        values.append(
            metric_function(
                labels[indices],
                predictions[indices],
            )
        )

    alpha = (
        1.0 - confidence_level
    ) / 2.0

    lower = float(
        np.quantile(values, alpha)
    )
    upper = float(
        np.quantile(
            values,
            1.0 - alpha,
        )
    )
    point = float(
        metric_function(
            labels,
            predictions,
        )
    )

    return point, lower, upper


accuracy_point, accuracy_lower, accuracy_upper = (
    bootstrap_metric_interval(
        test_metrics["labels"],
        test_metrics["predictions"],
        accuracy_score,
    )
)

f1_point, f1_lower, f1_upper = (
    bootstrap_metric_interval(
        test_metrics["labels"],
        test_metrics["predictions"],
        lambda left, right: f1_score(
            left,
            right,
            average="macro",
        ),
    )
)

pd.DataFrame(
    [
        (
            "Accuracy",
            accuracy_point,
            accuracy_lower,
            accuracy_upper,
        ),
        (
            "Macro F1",
            f1_point,
            f1_lower,
            f1_upper,
        ),
    ],
    columns=[
        "Metric",
        "Point estimate",
        "CI lower",
        "CI upper",
    ],
)

Confidence intervals may be wide on small test sets.

# 27. Paired Model Comparison

Since both models predict the same examples, comparison should preserve pairing.

In [ ]:
frozen_test = evaluate_model(
    trained_frozen_model,
    test_loader,
)

fine_tuned_test = evaluate_model(
    trained_fine_tuned_model,
    test_loader,
)


def paired_bootstrap_difference(
    labels: np.ndarray,
    predictions_a: np.ndarray,
    predictions_b: np.ndarray,
    metric_function,
    repetitions: int = 1000,
    seed: int = 42,
):
    generator = np.random.default_rng(
        seed
    )

    differences = []
    sample_size = len(labels)

    for _ in range(repetitions):
        indices = generator.integers(
            0,
            sample_size,
            size=sample_size,
        )

        score_a = metric_function(
            labels[indices],
            predictions_a[indices],
        )

        score_b = metric_function(
            labels[indices],
            predictions_b[indices],
        )

        differences.append(
            score_b - score_a
        )

    return {
        "observed_difference": float(
            metric_function(
                labels,
                predictions_b,
            )
            - metric_function(
                labels,
                predictions_a,
            )
        ),
        "ci_lower": float(
            np.quantile(
                differences,
                0.025,
            )
        ),
        "ci_upper": float(
            np.quantile(
                differences,
                0.975,
            )
        ),
    }


paired_result = paired_bootstrap_difference(
    frozen_test["labels"],
    frozen_test["predictions"],
    fine_tuned_test["predictions"],
    lambda left, right: f1_score(
        left,
        right,
        average="macro",
    ),
)

pd.Series(
    paired_result,
    name="Fine-tuned minus frozen macro F1",
)

If the interval crosses zero, the observed difference is not clearly separated from
sampling variability under this procedure.

# 28. Calibration Discussion

A calibrated classifier's confidence should correspond to empirical correctness.

Calibration methods include:

- reliability diagrams;
- expected calibration error;
- temperature scaling;
- isotonic regression.

Calibration must be estimated on validation data, not the test set.

# 29. Optional Hugging Face Setup

In [ ]:
TRANSFORMERS_AVAILABLE = (
    importlib.util.find_spec(
        "transformers"
    )
    is not None
)

DATASETS_AVAILABLE = (
    importlib.util.find_spec(
        "datasets"
    )
    is not None
)

RUN_HUGGING_FACE_DEMOS = False
USE_LOCAL_FILES_ONLY = True

MODEL_ID = "distilbert/distilbert-base-uncased"

pd.Series(
    {
        "transformers installed": TRANSFORMERS_AVAILABLE,
        "datasets installed": DATASETS_AVAILABLE,
        "run demos": RUN_HUGGING_FACE_DEMOS,
        "local files only": USE_LOCAL_FILES_ONLY,
        "model ID": MODEL_ID,
    }
)

Optional cells remain disabled by default so the notebook executes offline.

# 30. Optional Tokenization

In [ ]:
if (
    TRANSFORMERS_AVAILABLE
    and RUN_HUGGING_FACE_DEMOS
):
    from transformers import AutoTokenizer

    hf_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        local_files_only=(
            USE_LOCAL_FILES_ONLY
        ),
    )

    encoded_batch = hf_tokenizer(
        list(X_train.head(4)),
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    )

    print(
        "input_ids:",
        encoded_batch[
            "input_ids"
        ].shape,
    )
    print(
        "attention_mask:",
        encoded_batch[
            "attention_mask"
        ].shape,
    )
else:
    print(
        "Optional Hugging Face tokenization skipped."
    )

# 31. Optional AutoModel Fine-Tuning

In [ ]:
if (
    TRANSFORMERS_AVAILABLE
    and RUN_HUGGING_FACE_DEMOS
):
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        local_files_only=(
            USE_LOCAL_FILES_ONLY
        ),
    )

    hf_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_ID,
            num_labels=len(
                label_encoder.classes_
            ),
            id2label=id2label,
            label2id=label2id,
            local_files_only=(
                USE_LOCAL_FILES_ONLY
            ),
        )
        .to("cpu")
    )

    sample_inputs = tokenizer(
        list(X_train.head(2)),
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    )

    with torch.no_grad():
        sample_outputs = hf_model(
            **sample_inputs
        )

    print(
        "Logits:",
        sample_outputs.logits.shape,
    )
else:
    print(
        "Optional AutoModel fine-tuning setup skipped."
    )

# 32. Optional Trainer Workflow

The following code is shown as a version-sensitive template and is not executed.

In [ ]:
trainer_template = '''
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

arguments = TrainingArguments(
    output_dir="checkpoints/text-classification",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=arguments,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
'''

print(trainer_template)

Exact Trainer arguments should be verified against the installed Transformers
version.

# 33. Checkpoint Management

A complete checkpoint should preserve:

- model weights;
- tokenizer;
- configuration;
- label mappings;
- training arguments;
- validation metric;
- software versions;
- random seed.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = (
        Path(directory)
        / "classifier_checkpoint.pt"
    )

    torch.save(
        {
            "model_state_dict": (
                selected_model.state_dict()
            ),
            "vocabulary": vocabulary,
            "label2id": label2id,
            "id2label": id2label,
            "selected_model": selected_model_name,
            "validation_macro_f1": max(
                frozen_validation["macro_f1"],
                fine_tuned_validation["macro_f1"],
            ),
        },
        checkpoint_path,
    )

    loaded_checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

pd.Series(
    {
        "saved model": loaded_checkpoint[
            "selected_model"
        ],
        "saved vocabulary size": len(
            loaded_checkpoint[
                "vocabulary"
            ]
        ),
        "saved labels": len(
            loaded_checkpoint[
                "label2id"
            ]
        ),
    }
)

# 34. Learning-Rate and Batch-Size Decisions

Typical fine-tuning risks:

- learning rate too high: pretrained features are disrupted;
- learning rate too low: adaptation is weak;
- batch size too large: memory failure;
- batch size too small: noisy gradients.

Gradient accumulation can simulate a larger effective batch size.

In [ ]:
hyperparameter_tradeoffs = pd.DataFrame(
    [
        ("Learning rate", "adaptation speed", "representation damage"),
        ("Batch size", "throughput", "memory use"),
        ("Epoch count", "task fit", "overfitting"),
        ("Maximum length", "context retained", "quadratic cost"),
        ("Weight decay", "regularization", "underfitting"),
    ],
    columns=[
        "Hyperparameter",
        "Benefit when increased",
        "Risk when increased",
    ],
)

hyperparameter_tradeoffs

# 35. Class Imbalance

For imbalanced data, consider:

- macro F1;
- per-class metrics;
- class-weighted loss;
- resampling;
- threshold adjustment;
- stratified splitting.

In [ ]:
class_distribution = dataset[
    "label"
].value_counts().rename(
    "count"
).to_frame()

class_distribution

# 36. Domain Shift

A pretrained model may transfer poorly when the downstream domain differs from its
pretraining data.

Indicators:

- high unknown or fragmented-token rate;
- weak frozen baseline;
- large fine-tuning gain;
- unstable validation;
- systematic errors on domain terms.

# 37. Arabic and Multilingual Considerations

Arabic fine-tuning should examine:

- MSA versus dialect coverage;
- vocalized versus unvocalized text;
- clitic segmentation;
- orthographic normalization;
- subword fragmentation;
- class-label consistency across language varieties;
- cross-language imbalance.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "وَ + سَ + يَكْتُبُونَ + هَا",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "بِ + الْمَدْرَسَةِ",
        ),
        (
            "كِتَابُهُمَا",
            "كِتَابُ + هُمَا",
        ),
    ],
    columns=[
        "Fully vocalized form",
        "Illustrative segmentation",
    ],
)

arabic_examples

For fully vocalized Arabic tasks, tashkeel must remain in the tokenizer input when
it is part of the task definition. Removing it changes the data and may collapse
meaningful distinctions.

In [ ]:
multilingual_checks = pd.DataFrame(
    [
        ("Tokenizer coverage", "inspect fragmentation"),
        ("Language balance", "report examples per language"),
        ("Dialect balance", "separate or stratify varieties"),
        ("Tashkeel policy", "preserve or document removal"),
        ("Error analysis", "inspect morphology and clitics"),
    ],
    columns=["Check", "Action"],
)

multilingual_checks

# 38. Reproducibility and Reporting

Report:

- model ID and revision;
- tokenizer;
- dataset and split;
- label mappings;
- maximum length;
- truncation and padding;
- frozen or fine-tuned parameters;
- learning rates;
- batch size;
- epochs;
- checkpoint-selection metric;
- random seeds;
- software versions;
- hardware;
- test metrics and confidence intervals;
- limitations.

In [ ]:
reproducibility_metadata = pd.Series(
    {
        "selected strategy": selected_model_name,
        "training examples": len(X_train),
        "validation examples": len(X_validation),
        "test examples": len(X_test),
        "classes": len(label_encoder.classes_),
        "device": str(DEVICE),
        "seed": 42,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers installed": (
            TRANSFORMERS_AVAILABLE
        ),
        "optional demos enabled": (
            RUN_HUGGING_FACE_DEMOS
        ),
    },
    name="Lesson 36 experiment",
)

reproducibility_metadata

# 39. Knowledge Check

1. What is supervised fine-tuning?
2. Why is a validation set required?
3. Why should the test set be used only once?
4. What do label2id and id2label provide?
5. How does frozen transfer differ from full fine-tuning?
6. Why is the fine-tuning learning rate usually small?
7. What does macro F1 measure?
8. Why inspect per-class metrics?
9. What does a confusion matrix reveal?
10. Why is confidence not equivalent to correctness?
11. What does a bootstrap confidence interval estimate?
12. Why use paired resampling for model comparison?
13. What information belongs in a checkpoint?
14. What is domain shift?
15. How can Arabic tokenization affect classification?

# 40. Exercises

## Exercise 1 — New Dataset

Replace the synthetic data with a documented public classification dataset.

## Exercise 2 — Frozen Baseline

Train only the classification head of a pretrained encoder.

## Exercise 3 — Full Fine-Tuning

Fine-tune all encoder layers with a smaller learning rate.

## Exercise 4 — Gradual Unfreezing

Unfreeze one encoder layer at a time.

## Exercise 5 — Class Weights

Add class-weighted cross-entropy.

## Exercise 6 — Calibration

Create a reliability diagram and expected calibration error.

## Exercise 7 — Statistical Comparison

Compare two checkpoints with paired bootstrap resampling.

## Exercise 8 — Error Taxonomy

Build a manual taxonomy of classification errors.

## Exercise 9 — Arabic Fine-Tuning

Fine-tune an Arabic encoder on a fully vocalized MSA dataset.

## Exercise 10 — Model Card

Write a model card for the final classifier.

## Challenge Exercises

1. Add differential learning rates for encoder and classifier.
2. Add learning-rate warmup and decay.
3. Implement early stopping with the Trainer API.
4. Compare multilingual and Arabic-specific checkpoints.
5. Publish the tokenizer, model, metrics, and model card to a private Hub
   repository.

# 41. Summary and Next Lesson

In this lesson:

- fine-tuning was framed as supervised transfer learning;
- deterministic train, validation, and test splits were created;
- label mappings were defined and preserved;
- tokenization, padding, and truncation decisions were examined;
- frozen and fully fine-tuned Transformer classifiers were trained;
- validation macro F1 selected the final checkpoint;
- accuracy, macro F1, per-class metrics, confusion matrices, confidence, and
  errors were analyzed;
- bootstrap confidence intervals quantified metric uncertainty;
- paired bootstrap comparison evaluated frozen versus fine-tuned performance;
- checkpoint metadata and reproducibility requirements were preserved;
- optional Hugging Face AutoTokenizer, AutoModel, and Trainer workflows were
  provided;
- class imbalance, domain shift, Arabic morphology, multilingual data, and
  tashkeel were integrated into the fine-tuning workflow.

## Next Lesson

**Lesson 37: Token Classification with Pretrained Transformers** introduces
subword-label alignment, BIO tagging, named-entity recognition, padding-aware
token loss, span-level evaluation, and multilingual sequence labeling.

# References

- Hugging Face Transformers documentation: sequence classification, Auto Classes,
  data collators, Trainer, and model saving.
- Devlin, J. et al. BERT.
- Vaswani, A. et al. *Attention Is All You Need*.
- Efron, B., & Tibshirani, R. bootstrap methods.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.